# Langsmith - AI observability

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

load_dotenv()  #! load langsmith creds


class LMStudio:
    # Connection settings
    BASE_URL = "http://localhost:1234/v1"
    API_KEY = "lm-studio"
    MODEL = "model-identifier"

    # Behavior settings
    TEMPERATURE = 1  # 0 - 1, 0 is most accurate, 1 is most creative


llm = ChatOpenAI(
    model=LMStudio.MODEL,
    base_url=LMStudio.BASE_URL,
    api_key=SecretStr(LMStudio.API_KEY),
)

#! just by invoking this -> our project is registered in langsmith
llm.invoke("hello, whats your name and what are you?") 

AIMessage(content="Hello! My name is Gemma. I'm a large language model, created by the Gemma team at Google DeepMind. Basically, I'm an AI that can communicate and generate text in response to prompts and questions. \n\nI'm also an open-weights model, which means I'm widely available for people to use and build upon!\n\n\n\nIt's nice to meet you! 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 19, 'total_tokens': 101, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'gemma-3-12b-it', 'system_fingerprint': 'gemma-3-12b-it', 'id': 'chatcmpl-dtggg0d6srm9m1v2mrwfq', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--5b402423-b68a-41d0-8ec2-ea429f5cb61b-0', usage_metadata={'input_tokens': 19, 'output_tokens': 82, 'total_tokens': 101, 'input_token_details': {}, 'output_token_details': {}})

LangSmith can trace functions that are not part of LangChain, we just need to add the @traceable decorator. Let's try this for a few simple functions.

In [5]:
from langsmith import traceable
import random
import time
from tqdm.auto import tqdm


@traceable
def generate_random_number():
    return random.randint(0, 100)

@traceable
def generate_string_delay(input_str: str):
    number = random.randint(1, 5)
    time.sleep(number)
    return f"{input_str} ({number})"

@traceable
def random_error():
    number = random.randint(0, 1)
    if number == 0:
        raise ValueError("Random error")
    else:
        return "No error"
    
    
@traceable(name="Chitchat Maker")
def error_generation_function(question: str):
    delay = random.randint(0, 3)
    time.sleep(delay)
    number = random.randint(0, 1)
    if number == 0:
        raise ValueError("Random error")
    else:
        return "I'm great how are you?"
    
for _ in tqdm(range(10)):
    generate_random_number()
    generate_string_delay("Hello")
    
    try:
        random_error()
        error_generation_function("what is my name?")
    except ValueError:
        pass

100%|██████████| 10/10 [00:49<00:00,  4.91s/it]
